# 添正生物 · AI 测试模拟数据读取（Polars）

**数据源**：`副本AI 测试模拟数据-20260805(1).xlsx` → `Sheet1`（Sheet2 / Sheet3 为空）

## 表结构（已核实）

| 位置 | 内容 |
|---|---|
| 第 1 行 | 大标题 `AI测试模拟数据`（合并单元格 A1:L1）—— **不是表头** |
| 第 2 行 | 真正的表头 |
| 第 3 ~ 4057 行 | 数据区，其中**尾部 177 行是全空行**（Excel 里带格式但无内容）→ 有效数据 **3878 条** |
| A ~ X（24 列） | 业务原始检测指标 —— 分析主体 |
| Y ~ AP（18 列） | 原表的辅助/中间计算列，公式已失效；其中 `合格`/`月份`/`pi` 三列**整列 `#REF!`** |

## 读取时必须绕开的 4 个坑

1. **表头在第 2 行** —— 直接 `read_excel` 会把大标题当表头，整表报废
2. **列名重复** —— `大肠发酵`/`沙门发酵`/`大肠疑似`/`沙门疑似` 在 A~X 和 Y~AP 各出现一次，重名会让 DataFrame 建不出来
3. **混合类型** —— `菌落总数`=`<10`、`水不溶物`=`2级`、`大肠杆菌`=`未检出`、`霉菌`=`TNTC`。
   自动类型推断会把整列吃成 null，所以**先整表按字符串读入，再显式转换**
4. **检出限值** —— `<10` / `<1` / `TNTC` 是微生物检测的删失数据，
   不能简单丢弃，拆成「数值 + 算符」两列保留信息

## 1. 依赖与导入

In [35]:
# 首次运行如缺少依赖，取消下面一行的注释执行安装：
# %pip install polars fastexcel

import warnings
from pathlib import Path

import polars as pl

print("polars:", pl.__version__)

polars: 1.43.2


## 2. 配置

In [36]:
# 显示设置：宽表尽量完整展示
pl.Config.set_tbl_cols(60)
pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(40)
pl.Config.set_tbl_width_chars(220)

XLSX  = Path("副本AI 测试模拟数据-20260805(1).xlsx")
SHEET = "Sheet1"

assert XLSX.exists(), f"找不到文件：{XLSX.resolve()}"
print("文件：", XLSX.resolve())
print("大小：", f"{XLSX.stat().st_size / 1024 / 1024:.2f} MB")

文件： /Users/liyikang/Desktop/添正生物/副本AI 测试模拟数据-20260805(1).xlsx
大小： 1.63 MB


## 3. 整表读取

不使用自动表头（`has_header=False`），把整张表当字符串矩阵读进来：

- 第 0 行 = Excel 第 1 行（大标题）→ 丢弃
- 第 1 行 = Excel 第 2 行（表头）→ 取出来清洗成列名
- 第 2 行往后 = 数据

In [37]:
def clean_name(name: object, idx: int) -> str:
    # 表头清洗：去掉换行与多余空格，例如 '重量\nkg' -> '重量kg'
    s = "" if name is None else "".join(str(name).split())
    return s or f"col_{idx}"


def dedup(names: list[str]) -> list[str]:
    # 列名去重：重名的第 2 次出现加 __2 后缀
    seen, out = {}, []
    for n in names:
        if n in seen:
            seen[n] += 1
            out.append(f"{n}__{seen[n]}")
        else:
            seen[n] = 1
            out.append(n)
    return out


with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)   # polars/fastexcel 版本兼容提示，无影响
    raw = pl.read_excel(
        XLSX,
        sheet_name=SHEET,
        engine="calamine",
        has_header=False,
        read_options={"dtypes": "string"},   # 全部按字符串，保留 <10 / 未检出 / #REF! 原文
    )

title  = raw.row(0)[0]
header = dedup([clean_name(v, i) for i, v in enumerate(raw.row(1))])
df_all = raw.slice(2).rename(dict(zip(raw.columns, header)))

# 丢掉整列为空的列：Excel 里 AQ~AU 只有零星空白字符，合格/月份/pi 整列 #REF!（读入后为 null）
nonempty = [c for c in df_all.columns
            if df_all[c].str.strip_chars().replace("", None).null_count() < df_all.height]
dropped  = [c for c in df_all.columns if c not in nonempty]
df_all   = df_all.select(nonempty)

print("大标题  ：", title)
print("原始形状：", df_all.shape)
print("已丢弃的全空列：", dropped, "（合格/月份/pi 整列 #REF!，col_4x 为空白填充列）")
print("\n列名：")
for i, c in enumerate(df_all.columns, 1):
    print(f"  {i:>2}. {c}")

大标题  ： AI测试模拟数据
原始形状： (4055, 39)
已丢弃的全空列： ['合格', '月份', 'pi', 'col_42', 'col_43', 'col_44', 'col_45', 'col_46'] （合格/月份/pi 整列 #REF!，col_4x 为空白填充列）

列名：
   1. 数据编号
   2. 重量kg
   3. 冻力Bloomg
   4. 水分%
   5. 灰分%
   6. PH值
   7. 水不溶物（个）
   8. 勃氏粘度mPa/s
   9. 粘度下降%
  10. 透过率450%
  11. 透过率620%
  12. 电导率us/cm
  13. 二氧化硫mg/kg
  14. 过氧化物mg/kg
  15. 菌落总数
  16. 霉菌
  17. 大肠杆菌
  18. 沙门氏菌
  19. 大肠发酵
  20. 沙门发酵
  21. 大肠疑似
  22. 沙门疑似
  23. 高温菌计数
  24. 金葡球菌
  25. Bloom
  26. viscosity
  27. moisture
  28. residue
  29. PH
  30. T450
  31. T620
  32. 电导率
  33. 二氧化硫
  34. 大肠发酵__2
  35. 沙门发酵__2
  36. 大肠合格
  37. 沙门合格
  38. 大肠疑似__2
  39. 沙门疑似__2


## 4. 列分组：核心指标（A~X） vs 辅助计算列（Y~AP）

In [38]:
CORE_COLS = [
    "数据编号", "重量kg", "冻力Bloomg", "水分%", "灰分%", "PH值", "水不溶物（个）",
    "勃氏粘度mPa/s", "粘度下降%", "透过率450%", "透过率620%", "电导率us/cm",
    "二氧化硫mg/kg", "过氧化物mg/kg", "菌落总数", "霉菌", "大肠杆菌", "沙门氏菌",
    "大肠发酵", "沙门发酵", "大肠疑似", "沙门疑似", "高温菌计数", "金葡球菌",
]

core_cols = [c for c in CORE_COLS if c in df_all.columns]     # 与实际表头对齐
missing   = [c for c in CORE_COLS if c not in df_all.columns]
aux_cols  = [c for c in df_all.columns if c not in core_cols]

print(f"核心指标列 {len(core_cols)} 个")
if missing:
    print("⚠️ 未匹配到的列（请核对表头）：", missing)
print(f"辅助计算列 {len(aux_cols)} 个：", aux_cols)

核心指标列 24 个
辅助计算列 15 个： ['Bloom', 'viscosity', 'moisture', 'residue', 'PH', 'T450', 'T620', '电导率', '二氧化硫', '大肠发酵__2', '沙门发酵__2', '大肠合格', '沙门合格', '大肠疑似__2', '沙门疑似__2']


### 4.1 剔除空行

表尾 177 行「有格式无内容」：A~X 全空，只有辅助公式列对空行算出一串 `0`。
所以**空行判定只看核心列**，否则一行都删不掉，缺失率/均值/计数全被拉偏。

In [39]:
core_blank = pl.all_horizontal(
    pl.col(c).str.strip_chars().replace("", None).is_null() for c in core_cols
)

n_before = df_all.height
df_all   = df_all.filter(~core_blank)

df_core = df_all.select(core_cols)   # 业务指标
df_aux  = df_all.select(aux_cols)    # 辅助列

print(f"剔除空行：{n_before} → {df_all.height}（删除 {n_before - df_all.height} 行）")
print(f"有效数据：{df_core.height} 条 × {df_core.width} 列")

剔除空行：4055 → 3878（删除 177 行）
有效数据：3878 条 × 24 列


## 5. 类型转换

- **纯数值列** → `Float64`（`数据编号` → `Int64`）
- **检出限列**（`菌落总数` / `霉菌` / `高温菌计数`）→ 第 6 节单独拆解，此处保持原文
- **判定列**（`2级` / `未检出` / `否` / `未见`）→ 保持字符串

`strict=False`：转不了的值变 `null`，不会中断。

In [40]:
NUMERIC_COLS = [
    "重量kg", "冻力Bloomg", "水分%", "灰分%", "PH值",
    "勃氏粘度mPa/s", "粘度下降%", "透过率450%", "透过率620%",
    "电导率us/cm", "二氧化硫mg/kg", "过氧化物mg/kg",
]
CENSORED_COLS = ["菌落总数", "霉菌", "高温菌计数"]   # 含 <10 / <1 / TNTC

num_cols = [c for c in NUMERIC_COLS if c in df_core.columns]
cat_cols = [c for c in df_core.columns
            if c not in num_cols + CENSORED_COLS + ["数据编号"]]

df = df_core.with_columns(
    [pl.col("数据编号").str.strip_chars().cast(pl.Int64, strict=False)]
    + [pl.col(c).str.strip_chars().cast(pl.Float64, strict=False) for c in num_cols]
    + [pl.col(c).str.strip_chars() for c in cat_cols + CENSORED_COLS]
)

# 转换损耗自检：原本有值、转完变 null 的数量
loss = {
    c: int(df_core[c].str.strip_chars().replace("", None).is_not_null().sum()
           - df[c].is_not_null().sum())
    for c in ["数据编号"] + num_cols
}
bad = {k: v for k, v in loss.items() if v}

print("数值列  ：", num_cols)
print("检出限列：", CENSORED_COLS)
print("判定列  ：", cat_cols)
print("转换损耗：", bad if bad else "无（数值列全部干净转换）")

数值列  ： ['重量kg', '冻力Bloomg', '水分%', '灰分%', 'PH值', '勃氏粘度mPa/s', '粘度下降%', '透过率450%', '透过率620%', '电导率us/cm', '二氧化硫mg/kg', '过氧化物mg/kg']
检出限列： ['菌落总数', '霉菌', '高温菌计数']
判定列  ： ['水不溶物（个）', '大肠杆菌', '沙门氏菌', '大肠发酵', '沙门发酵', '大肠疑似', '沙门疑似', '金葡球菌']
转换损耗： 无（数值列全部干净转换）


## 6. 检出限列拆解

微生物指标的 `<10`、`<1` 表示「低于检出限」，`TNTC` = too numerous to count（多到无法计数）。
直接 `cast` 会把这些值全变 `null`（`高温菌计数` 会丢掉 670 个 `<1`）。

拆成两列保留全部信息：

| 新列 | 含义 |
|---|---|
| `xxx_数值` | 数值部分，`<10` → `10.0`，`TNTC` → `null` |
| `xxx_算符` | `<`（低于检出限） / `=`（实测值） / `TNTC` |

建模时：`算符 == "<"` 的样本按左删失处理，或统一用 `数值` 作保守上界。

In [41]:
df = df.with_columns(
    [
        pl.col(c).str.replace(r"^<", "").cast(pl.Float64, strict=False).alias(f"{c}_数值")
        for c in CENSORED_COLS
    ]
    + [
        pl.when(pl.col(c).is_null()).then(None)
          .when(pl.col(c).str.starts_with("<")).then(pl.lit("<"))
          .when(pl.col(c).str.contains(r"^\d")).then(pl.lit("="))
          .otherwise(pl.col(c))
          .alias(f"{c}_算符")
        for c in CENSORED_COLS
    ]
)

for c in CENSORED_COLS:
    print(f"─── {c} ───")
    print(df[f"{c}_算符"].value_counts(sort=True))

─── 菌落总数 ───
shape: (3, 2)
┌───────────────┬───────┐
│ 菌落总数_算符 ┆ count │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ <             ┆ 3416  │
│ =             ┆ 450   │
│ TNTC          ┆ 12    │
└───────────────┴───────┘
─── 霉菌 ───
shape: (3, 2)
┌───────────┬───────┐
│ 霉菌_算符 ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ <         ┆ 3590  │
│ =         ┆ 274   │
│ TNTC      ┆ 14    │
└───────────┴───────┘
─── 高温菌计数 ───
shape: (3, 2)
┌─────────────────┬───────┐
│ 高温菌计数_算符 ┆ count │
│ ---             ┆ ---   │
│ str             ┆ u32   │
╞═════════════════╪═══════╡
│ null            ┆ 3103  │
│ <               ┆ 670   │
│ =               ┆ 105   │
└─────────────────┴───────┘


## 7. 整表信息总览

In [42]:
print(f"=== 数据规模：{df.height} 行 × {df.width} 列 ===\n")
print("=== Schema ===")
for c, t in df.schema.items():
    print(f"  {c:<20} {t}")

=== 数据规模：3878 行 × 30 列 ===

=== Schema ===
  数据编号                 Int64
  重量kg                 Float64
  冻力Bloomg             Float64
  水分%                  Float64
  灰分%                  Float64
  PH值                  Float64
  水不溶物（个）              String
  勃氏粘度mPa/s            Float64
  粘度下降%                Float64
  透过率450%              Float64
  透过率620%              Float64
  电导率us/cm             Float64
  二氧化硫mg/kg            Float64
  过氧化物mg/kg            Float64
  菌落总数                 String
  霉菌                   String
  大肠杆菌                 String
  沙门氏菌                 String
  大肠发酵                 String
  沙门发酵                 String
  大肠疑似                 String
  沙门疑似                 String
  高温菌计数                String
  金葡球菌                 String
  菌落总数_数值              Float64
  霉菌_数值                Float64
  高温菌计数_数值             Float64
  菌落总数_算符              String
  霉菌_算符                String
  高温菌计数_算符             String


In [43]:
df.head(10)

数据编号,重量kg,冻力Bloomg,水分%,灰分%,PH值,水不溶物（个）,勃氏粘度mPa/s,粘度下降%,透过率450%,透过率620%,电导率us/cm,二氧化硫mg/kg,过氧化物mg/kg,菌落总数,霉菌,大肠杆菌,沙门氏菌,大肠发酵,沙门发酵,大肠疑似,沙门疑似,高温菌计数,金葡球菌,菌落总数_数值,霉菌_数值,高温菌计数_数值,菌落总数_算符,霉菌_算符,高温菌计数_算符
i64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,f64,f64,f64,str,str,str
1,216.0,163.1,12.65,0.27,5.78,"""2级""",5.66,1.94,86.0,96.9,124.1,6.4,0.0,"""<10""","""<10""","""未检出""","""未检出""","""否""","""微""","""未见""","""未见""","""1""","""未检出""",10.0,10.0,1.0,"""<""","""<""","""="""
2,512.0,273.1,12.96,0.27,5.78,"""2级""",5.68,1.94,87.3,97.3,124.1,6.4,0.0,"""<10""","""<10""","""未检出""","""未检出""","""否""","""否""","""未见""","""未见""",null,null,10.0,10.0,null,"""<""","""<""",null
3,516.2,274.7,12.88,0.27,5.78,"""2级""",5.66,1.94,87.5,97.5,124.1,6.4,0.0,"""<10""","""<10""","""未检出""","""未检出""","""否""","""否""","""未见""","""未见""",null,null,10.0,10.0,null,"""<""","""<""",null
4,519.2,273.3,12.91,0.27,5.78,"""2级""",5.68,1.94,88.1,97.8,124.1,6.4,0.0,"""<10""","""<10""","""未检出""","""未检出""","""否""","""微""","""未见""","""未见""",null,null,10.0,10.0,null,"""<""","""<""",null
5,504.2,275.2,12.86,0.27,5.78,"""2级""",5.68,1.94,88.3,97.8,124.1,6.4,0.0,"""<10""","""<10""","""未检出""","""未检出""","""否""","""是""","""未见""","""未见""",null,null,10.0,10.0,null,"""<""","""<""",null
6,518.2,273.4,12.69,0.27,5.78,"""2级""",5.52,1.94,88.6,97.9,124.1,6.4,0.0,"""<10""","""<10""","""未检出""","""未检出""","""否""","""是""","""未见""","""未见""",null,null,10.0,10.0,null,"""<""","""<""",null
7,518.6,269.4,12.58,0.27,5.78,"""2级""",5.57,1.94,88.7,98.0,124.1,6.4,0.0,"""<10""","""<10""","""未检出""","""未检出""","""否""","""否""","""未见""","""未见""",null,null,10.0,10.0,null,"""<""","""<""",null
8,515.8,265.0,12.41,0.27,5.78,"""2级""",5.6,1.94,88.1,97.6,124.1,6.4,0.0,"""<10""","""<10""","""未检出""","""未检出""","""否""","""否""","""未见""","""未见""",null,null,10.0,10.0,null,"""<""","""<""",null
9,541.8,259.4,12.24,0.27,5.78,"""2级""",5.87,1.94,85.4,96.6,124.1,6.4,0.0,"""<10""","""<10""","""未检出""","""未检出""","""否""","""否""","""未见""","""未见""",null,null,10.0,10.0,null,"""<""","""<""",null


## 8. 缺失值统计

In [44]:
(
    df.null_count()
      .transpose(include_header=True, header_name="列名", column_names=["缺失数"])
      .with_columns((pl.col("缺失数") / df.height * 100).round(2).alias("缺失率%"))
      .sort("缺失数", descending=True)
)

列名,缺失数,缺失率%
str,u32,f64
"""金葡球菌""",3104,80.04
"""高温菌计数""",3103,80.02
"""高温菌计数_数值""",3103,80.02
"""高温菌计数_算符""",3103,80.02
"""霉菌_数值""",14,0.36
"""菌落总数_数值""",12,0.31
"""数据编号""",0,0.0
"""重量kg""",0,0.0
"""冻力Bloomg""",0,0.0


## 9. 数值列描述统计

In [45]:
df.select(num_cols + [f"{c}_数值" for c in CENSORED_COLS]).describe()

statistic,重量kg,冻力Bloomg,水分%,灰分%,PH值,勃氏粘度mPa/s,粘度下降%,透过率450%,透过率620%,电导率us/cm,二氧化硫mg/kg,过氧化物mg/kg,菌落总数_数值,霉菌_数值,高温菌计数_数值
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",3878.0,3878.0,3878.0,3878.0,3878.0,3878.0,3878.0,3878.0,3878.0,3878.0,3878.0,3878.0,3866.0,3864.0,775.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0,14.0,3103.0
"""mean""",497.146364,202.873442,11.476166,0.38189,5.656202,4.297573,1.602607,82.47411,95.260263,110.366916,9.492806,0.0,13.72478,10.872153,1.412903
"""std""",162.562992,61.250671,0.714211,0.143883,0.16291,0.925107,0.915651,8.041528,3.503384,16.483521,2.645203,0.0,92.736134,15.236175,3.328654
"""min""",63.0,59.8,8.68,0.05,4.97,1.83,0.0,27.2,61.5,36.9,2.1,0.0,10.0,10.0,0.0
"""25%""",407.4,146.9,11.06,0.27,5.56,3.58,0.91,77.4,93.7,100.6,7.6,0.0,10.0,10.0,1.0
"""50%""",499.0,214.9,11.47,0.37,5.68,4.46,1.4,83.4,96.1,111.0,9.4,0.0,10.0,10.0,1.0
"""75%""",545.0,256.6,11.87,0.47,5.77,5.0,2.13,89.5,98.0,120.8,11.2,0.0,10.0,10.0,1.0
"""max""",934.2,315.4,14.67,1.04,6.09,6.64,4.57,96.5,99.6,177.3,20.4,0.0,3960.0,750.0,58.0


## 10. 判定列取值分布

In [46]:
for c in cat_cols:
    vc = df[c].value_counts(sort=True)
    print(f"\n─── {c}（{vc.height} 种取值）───")
    print(vc.head(10))


─── 水不溶物（个）（4 种取值）───
shape: (4, 2)
┌────────────────┬───────┐
│ 水不溶物（个） ┆ count │
│ ---            ┆ ---   │
│ str            ┆ u32   │
╞════════════════╪═══════╡
│ 2级            ┆ 2223  │
│ 3级            ┆ 1554  │
│ 4级            ┆ 94    │
│ 1级            ┆ 7     │
└────────────────┴───────┘

─── 大肠杆菌（2 种取值）───
shape: (2, 2)
┌──────────┬───────┐
│ 大肠杆菌 ┆ count │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ 未检出   ┆ 3847  │
│ 检出     ┆ 31    │
└──────────┴───────┘

─── 沙门氏菌（1 种取值）───
shape: (1, 2)
┌──────────┬───────┐
│ 沙门氏菌 ┆ count │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ 未检出   ┆ 3878  │
└──────────┴───────┘

─── 大肠发酵（3 种取值）───
shape: (3, 2)
┌──────────┬───────┐
│ 大肠发酵 ┆ count │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ 否       ┆ 3768  │
│ 是       ┆ 90    │
│ 微       ┆ 20    │
└──────────┴───────┘

─── 沙门发酵（3 种取值）───
shape: (3, 2)
┌──────────┬───────┐
│ 沙门发酵 ┆ count │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════

## 11. 辅助列速览（Y~AP）

原表的中间计算列，公式已失效（`合格`/`月份`/`pi` 整列 `#REF!`，读入后为 null 已被丢弃）。
留档用，**不建议直接进模型**。

In [47]:
print("辅助列：", df_aux.columns)
df_aux.head(5)

辅助列： ['Bloom', 'viscosity', 'moisture', 'residue', 'PH', 'T450', 'T620', '电导率', '二氧化硫', '大肠发酵__2', '沙门发酵__2', '大肠合格', '沙门合格', '大肠疑似__2', '沙门疑似__2']


Bloom,viscosity,moisture,residue,PH,T450,T620,电导率,二氧化硫,大肠发酵__2,沙门发酵__2,大肠合格,沙门合格,大肠疑似__2,沙门疑似__2
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""35229.6""","""1222.56""","""2732.4""","""58.32""","""1248.48""","""18576""","""20930.4""","""26805.6""","""1382.4""","""216""","""""","""216""","""216""","""216""","""216"""
"""139827.2""","""2908.16""","""6635.52""","""138.24""","""2959.36""","""44697.6""","""49817.6""","""63539.2""","""3276.8""","""512""","""512""","""512""","""512""","""512""","""512"""
"""141800.14""","""2921.692""","""6648.656""","""139.374""","""2983.636""","""45167.5""","""50329.5""","""64060.42""","""3303.68""","""516.2""","""516.2""","""516.2""","""516.2""","""516.2""","""516.2"""
"""141897.36""","""2949.056""","""6702.872""","""140.184""","""3000.976""","""45741.52""","""50777.76""","""64432.72""","""3322.88""","""519.2""","""""","""519.2""","""519.2""","""519.2""","""519.2"""
"""138755.84""","""2863.856""","""6484.012""","""136.134""","""2914.276""","""44520.86""","""49310.76""","""62571.22""","""3226.88""","""504.2""","""""","""504.2""","""504.2""","""504.2""","""504.2"""


## 12. 落盘缓存

转成 Parquet 后，后续分析加载速度是读 Excel 的几十倍，且类型已固化。

In [48]:
OUT = Path("data_core.parquet")
df.write_parquet(OUT)
print("已保存：", OUT.resolve(), f"({OUT.stat().st_size / 1024:.0f} KB)")

# 后续直接用：
#   df = pl.read_parquet("data_core.parquet")
# 或惰性查询（大数据量推荐）：
#   lf = pl.scan_parquet("data_core.parquet")
#   lf.filter(pl.col("冻力Bloomg") > 250).select("数据编号", "冻力Bloomg").collect()

已保存： /Users/liyikang/Desktop/添正生物/data_core.parquet (85 KB)


---
# 13. 配料优化：按客户规格从半成品中选料

## 问题

客户给一张规格单，例如：

| 指标 | 要求 |
|---|---|
| 重量 | 200 kg ← **总量**，不是平均值 |
| 冻力 | 210 Bloomg |
| 水分 | 12 % |
| 灰分 | 0.25 % |
| PH | 6.0 |

从 3878 批半成品里挑若干批、各取多少，使**按重量加权平均**命中规格。

## 数学形式

设第 $i$ 批取用 $x_i$ kg（$0 \le x_i \le$ 库存量），$v_{ij}$ 为第 $i$ 批第 $j$ 项指标：

$$\sum_i x_i = W \qquad\qquad \left|\ \frac{\sum_i x_i v_{ij}}{W} - t_j\ \right| \le \text{tol}_j$$

分母 $\sum x_i$ 已固定为 $W$，所以加权平均约束是**线性**的 —— 可以用混合整数线性规划精确求解，
不需要启发式搜索。再引入 0/1 变量 $y_i$（该批是否启用）就能控制批次数。

## 两个求解目标

| 模式 | 目标 | 适用 |
|---|---|---|
| `fewest` | 批次数最少 | 减少投料/清洗工作量，容差内即可 |
| `closest` | 最大归一化偏差最小 | 规格卡得紧，要尽量居中 |

> **注意**：只有客户规格里出现的指标才会进约束。其余列（微生物、透过率等）
> 不参与计算，符合「一开始不会用到所有数值列」。

In [49]:
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds


def blend(pool, W, spec, *, mode="closest", min_take=0.0, max_batches=None,
          candidates=400, time_limit=30, weight_col="重量kg", id_col="数据编号"):
    # 配料求解
    #   pool        : 半成品 DataFrame
    #   W           : 目标总重量
    #   spec        : {指标列名: (目标值, 允许偏差)}
    #   mode        : "fewest" 批次最少 / "closest" 最贴合目标
    #   min_take    : 单批最小取用量，避免解出 0.3kg 这种没法执行的碎量
    #   max_batches : 批次数上限
    #   candidates  : 预筛候选数（closest 模式必需，否则 MILP 会很慢）
    cols = list(spec)
    p = pool.filter(pl.all_horizontal([pl.col(c).is_not_null() for c in cols + [weight_col]]))
    if p.height == 0:
        return {"status": "候选池为空（所需指标全部缺失）"}

    cap = p[weight_col].to_numpy().astype(float)
    V   = {c: p[c].to_numpy().astype(float) for c in cols}
    ids = p[id_col].to_numpy()

    # ---- 可行性预检：加权平均必落在池内极值之间，先挡掉无解情况 ----
    if cap.sum() < W:
        return {"status": f"库存不足：可用 {cap.sum():.1f} < 需求 {W}"}
    for c, (t, tol) in spec.items():
        lo, hi = V[c].min(), V[c].max()
        if t + tol < lo or t - tol > hi:
            return {"status": f"{c} 目标 {t}±{tol} 超出库存可达区间 [{lo:.3f}, {hi:.3f}]"}

    # ---- 候选预筛：按归一化距离取最近的 N 批 ----
    dist = sum(((V[c] - t) / tol) ** 2 for c, (t, tol) in spec.items())
    idx  = np.argsort(dist)[:candidates] if (candidates and len(cap) > candidates) \
           else np.arange(len(cap))
    cap, ids = cap[idx], ids[idx]
    V = {c: v[idx] for c, v in V.items()}
    n = len(cap)
    Z, I = np.zeros(n), np.eye(n)

    # ---- 变量：x(n 取用量) | y(n 是否启用) | [t(最大归一化偏差)] ----
    nt   = 1 if mode == "closest" else 0
    pad  = np.zeros((n, nt))
    rows, lb, ub = [np.r_[np.ones(n), Z, np.zeros(nt)]], [W], [W]   # Σx = W

    for c, (t, tol) in spec.items():
        if mode == "closest":                    # |Σx·v - t·W| <= tvar·tol·W
            rows += [np.r_[V[c], Z, -tol * W], np.r_[V[c], Z, tol * W]]
            lb   += [-np.inf, t * W]
            ub   += [t * W,   np.inf]
        else:                                    # 硬容差带
            rows.append(np.r_[V[c], Z])
            lb.append((t - tol) * W)
            ub.append((t + tol) * W)

    cons = [LinearConstraint(np.vstack(rows), lb, ub),
            LinearConstraint(np.hstack([I, -np.diag(cap), pad]), -np.inf, 0)]   # x <= cap·y
    if min_take > 0:
        cons.append(LinearConstraint(np.hstack([I, -min_take * I, pad]), 0, np.inf))
    if max_batches:
        cons.append(LinearConstraint(np.r_[Z, np.ones(n), np.zeros(nt)], 0, max_batches))

    cost = np.r_[Z, np.ones(n)] if mode == "fewest" else np.r_[Z, 1e3 * np.ones(n), 1.0]
    res = milp(
        c=cost, constraints=cons,
        integrality=np.r_[Z, np.ones(n), np.zeros(nt)],
        bounds=Bounds(np.zeros(2 * n + nt),
                      np.r_[cap, np.ones(n), np.full(nt, 1.0)]),
        options={"time_limit": time_limit},
    )
    if res.x is None:
        return {"status": f"无解：{res.message}"}

    x = res.x[:n]
    u = np.where(x > 1e-6)[0]
    picks = pl.DataFrame({
        id_col:  ids[u].astype(int),
        "取用kg": np.round(x[u], 2),
        "库存kg": np.round(cap[u], 1),
        "占比%":  np.round(x[u] / x.sum() * 100, 2),
        **{c: np.round(V[c][u], 3) for c in cols},
    }).sort("取用kg", descending=True)

    achieved = pl.DataFrame({
        "指标":   cols,
        "目标":   [spec[c][0] for c in cols],
        "实际":   [round(float(V[c] @ x / x.sum()), 4) for c in cols],
        "偏差":   [round(float(V[c] @ x / x.sum() - spec[c][0]), 4) for c in cols],
        "容差":   [spec[c][1] for c in cols],
        "占容差": [round(abs(float(V[c] @ x / x.sum() - spec[c][0])) / spec[c][1], 3) for c in cols],
    })

    return {"status": "OK", "批次数": len(u), "总量": round(float(x.sum()), 2),
            "用料": picks, "达成": achieved,
            "最大偏差占容差": float(achieved["占容差"].max())}

### 13.1 定义客户规格并求解

In [50]:
W = 200.0                       # 目标总重量 kg

SPEC = {                        # {指标: (目标值, 允许偏差)} —— 只写客户提的指标
    "冻力Bloomg": (210.0, 5.0),
    "水分%":      (12.0,  0.3),
    "灰分%":      (0.25,  0.05),
    "PH值":       (6.0,   0.2),
}

r = blend(df, W, SPEC, mode="closest", min_take=20.0, candidates=400)

print("状态：", r["status"])
if r["status"] == "OK":
    print(f"批次数：{r['批次数']}   总量：{r['总量']} kg   最大偏差：{r['最大偏差占容差']:.1%} 容差")

TypeError: Series constructor called with unsupported type 'ndarray' for the `values` parameter

### 13.2 用料单

In [51]:
r["用料"]

NameError: name 'r' is not defined

### 13.3 规格达成情况

In [52]:
r["达成"]

NameError: name 'r' is not defined

### 13.4 两种模式对比

`fewest` 只要求落在容差带内，解出来常常贴着容差边界；
`closest` 会把各项指标往目标中心拉。批次数相同时优先用 `closest`。

In [53]:
for mode in ("fewest", "closest"):
    res = blend(df, W, SPEC, mode=mode, min_take=20.0, candidates=400)
    if res["status"] != "OK":
        print(f"【{mode}】{res['status']}")
        continue
    print(f"\n【{mode}】批次数 {res['批次数']}，最大偏差 {res['最大偏差占容差']:.1%} 容差")
    print(res["达成"])

TypeError: Series constructor called with unsupported type 'ndarray' for the `values` parameter

### 13.5 可行性边界

规格无解时，先看是哪一项超出了库存的可达区间 —— 加权平均永远落在池内极值之间，
目标超出这个区间就**再怎么配也配不出来**，只能放宽容差或补料。

In [ ]:
spec_cols = list(SPEC)
avail = df.filter(pl.all_horizontal([pl.col(c).is_not_null() for c in spec_cols + ["重量kg"]]))

print(f"可用批次 {avail.height} 批，库存合计 {avail['重量kg'].sum():,.1f} kg\n")
pl.DataFrame({
    "指标":     spec_cols,
    "库存最小": [round(float(avail[c].min()), 3) for c in spec_cols],
    "库存最大": [round(float(avail[c].max()), 3) for c in spec_cols],
    "目标":     [SPEC[c][0] for c in spec_cols],
    "可达":     ["是" if avail[c].min() <= SPEC[c][0] <= avail[c].max() else "否"
                 for c in spec_cols],
})